# Projeto 2 — Sprint 2

## Teste de hipótese da média

**Variável:** Tempo de carregamento (min)  
**Planilha 1:** 300 observações antes da intervenção.  
**Planilha 2:** 100 observações depois da intervenção.

Este notebook segue a organização e o estilo dos notebooks da Sprint 1. Coloque os dois arquivos Excel na sessão do Colab antes de executar as células.


## 1. Bibliotecas e carregamento dos dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from pathlib import Path

def localizar_arquivo(caminho_repositorio, nome_upload):
    opcoes = [
        Path(caminho_repositorio),
        Path("../") / caminho_repositorio,
        Path("../../") / caminho_repositorio,
        Path(nome_upload)
    ]
    for caminho in opcoes:
        if caminho.exists():
            return str(caminho)
    raise FileNotFoundError(
        f"Arquivo não encontrado: {nome_upload}. "
        "Execute o notebook a partir da pasta principal do repositório "
        "ou envie a planilha para a sessão do Colab."
    )

ARQUIVO_ANTES = localizar_arquivo(
    "dados/planilha-1-antes-intervencao.xlsx",
    "Planilha1_Tempo de carregamento de caminhões com grãos.xlsx"
)
ARQUIVO_DEPOIS = localizar_arquivo(
    "dados/planilha-2-depois-intervencao.xlsx",
    "Planilha2_Tempo de carregamento de caminhões com grãos.xlsx"
)
COLUNA = "Tempo de carregamento (min)"

df_antes = pd.read_excel(ARQUIVO_ANTES)
df_depois = pd.read_excel(ARQUIVO_DEPOIS)

antes = pd.to_numeric(df_antes[COLUNA], errors="coerce").dropna()
depois = pd.to_numeric(df_depois[COLUNA], errors="coerce").dropna()

print("Antes:", len(antes), "observações")
print("Depois:", len(depois), "observações")

## Questão 1 — Comparação inicial dos resultados

In [ ]:
media_antes = antes.mean()
dp_antes = antes.std(ddof=1)
media_depois = depois.mean()
dp_depois = depois.std(ddof=1)

print(f"Média antes: {media_antes:.4f} minutos")
print(f"Média depois: {media_depois:.4f} minutos")
print(f"Desvio-padrão antes: {dp_antes:.4f} minutos")
print(f"Desvio-padrão depois: {dp_depois:.4f} minutos")
print(f"Redução observada: {media_antes - media_depois:.4f} minutos")

**Interpretação preliminar:** na Sprint 1, a média passou de aproximadamente 31,89 para 29,31 minutos. Essa diferença é descritiva e, sozinha, não comprova que a intervenção causou a redução.

## Questão 2a — Formulação das hipóteses

A média da Planilha 1 será utilizada como valor de referência. Como o objetivo é verificar se o tempo médio diminuiu, o teste é unilateral à esquerda:

- **H₀:** μ = μ₀
- **H₁:** μ < μ₀

Em que μ₀ é a média calculada com a Planilha 1.

In [ ]:
mu0 = media_antes
xbarra = media_depois
n = len(depois)

# A Planilha 1 fornece um desvio-padrão amostral.
# Ele será usado como estimativa de referência para o cálculo de z.
sigma = dp_antes

z_calculado = (xbarra - mu0) / (sigma / np.sqrt(n))

print(f"Valor de referência (mu0): {mu0:.4f}")
print(f"Média depois (xbarra): {xbarra:.4f}")
print(f"Desvio-padrão de referência: {sigma:.4f}")
print(f"Tamanho da amostra depois: {n}")
print(f"Valor de z calculado: {z_calculado:.4f}")

## Questão 2b — Teste com nível de significância de 5%

In [ ]:
alpha = 0.05
z_critico_5 = stats.norm.ppf(alpha)

print(f"Valor crítico: {z_critico_5:.4f}")
print(f"Região de rejeição: z <= {z_critico_5:.4f}")
print(f"Região de não rejeição: z > {z_critico_5:.4f}")

if z_calculado <= z_critico_5:
    print("Decisão: rejeitar H0.")
    print("Conclusão: há evidências de que a média depois da intervenção é menor.")
else:
    print("Decisão: não rejeitar H0.")
    print("Conclusão: não há evidências suficientes de que a média depois da intervenção é menor.")

In [ ]:
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x)

plt.figure(figsize=(9, 5))
plt.plot(x, y)
plt.fill_between(x, 0, y, where=(x <= z_critico_5), alpha=0.3,
                 label="Região de rejeição")
plt.axvline(z_critico_5, color="red", linestyle="--",
            label=f"Valor crítico = {z_critico_5:.2f}")
plt.axvline(z_calculado, color="green", linestyle=":",
            label=f"z calculado = {z_calculado:.2f}")
plt.title("Teste de hipótese da média — 5%")
plt.xlabel("z")
plt.ylabel("Densidade")
plt.legend()
plt.show()

## Teste com nível de significância de 1%

In [ ]:
alpha = 0.01
z_critico_1 = stats.norm.ppf(alpha)

print(f"Valor crítico: {z_critico_1:.4f}")
print(f"Região de rejeição: z <= {z_critico_1:.4f}")
print(f"Região de não rejeição: z > {z_critico_1:.4f}")

if z_calculado <= z_critico_1:
    print("Decisão: rejeitar H0.")
    print("Conclusão: há evidências de que a média depois da intervenção é menor.")
else:
    print("Decisão: não rejeitar H0.")
    print("Conclusão: não há evidências suficientes de que a média depois da intervenção é menor.")

In [ ]:
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x)

plt.figure(figsize=(9, 5))
plt.plot(x, y)
plt.fill_between(x, 0, y, where=(x <= z_critico_1), alpha=0.3,
                 label="Região de rejeição")
plt.axvline(z_critico_1, color="red", linestyle="--",
            label=f"Valor crítico = {z_critico_1:.2f}")
plt.axvline(z_calculado, color="green", linestyle=":",
            label=f"z calculado = {z_calculado:.2f}")
plt.title("Teste de hipótese da média — 1%")
plt.xlabel("z")
plt.ylabel("Densidade")
plt.legend()
plt.show()

## Questão 2c — Comparação com a análise preliminar

Compare as decisões dos testes de 1% e 5% com a redução descritiva observada na Questão 1.

Se a hipótese nula for rejeitada, há evidência estatística de redução da média no nível de significância utilizado. Se a hipótese nula não for rejeitada, isso significa que não houve evidência suficiente para afirmar a redução.

**Observação:** a Aula 2 apresenta a fórmula do teste z usando o desvio-padrão populacional. Como esse valor não foi fornecido nas planilhas, este notebook utiliza o desvio-padrão amostral da Planilha 1 como referência. Essa suposição deve ser informada na entrega e confirmada com o professor.